# Complete results for all four fracture samples

This notebook is the single entry point for every available per-sample HPC/Rorqual result and the original publication campaign. It draws BBFast, Mohr–Coulomb (MC), mesh variants, cyclic, shut-in, rate-and-state, and parameter-study simulations whenever their CSV histories are present.

Each sample has a dedicated block below. `MANUAL_RESULT_FILES` defaults to `None` for all four samples, so every discovered result is plotted automatically. Replace `None` with a list of filenames or stems only when a smaller comparison is wanted. All registered scalar files remain available through `POINT_DATA` and `point_result(...)`.

The loader discovers every CSV in each sample's `results_csv_hpc*` directory, prefers those files over the legacy campaign outputs, and chooses the longest readable duplicate. A run is marked **complete** only when its CSV reaches the **end_time** configured in its input deck. Partial files remain visible and are plotted to their actual end time.


In [ ]:
from pathlib import Path
import math
import re
import warnings

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 200)

# Set this to False for a quick core-results pass.
PLOT_ALL_SCALAR_COLUMNS = True
SCALAR_PLOTS_PER_FIGURE = 12
COMPLETE_TOLERANCE = 1.0e-6
SAMPLES = ["SWT1", "SWT2", "SWS3", "SWS4"]

# -----------------------------------------------------------------------------
# RESULT PLOT SELECTION
# None plots every registered result discovered for the sample. This includes
# BBFast, MC, mesh variants, cyclic, shut-in, rate-and-state, and parameter
# studies. Replace None with a list of CSV filenames/stems to make a subset.
# -----------------------------------------------------------------------------
MANUAL_RESULT_FILES = {sample: None for sample in SAMPLES}

REL_STUDY_DIR = Path("Examples/YeGhasemmi2018")


def locate_study_directory():
    cwd = Path.cwd().resolve()
    for anchor in (cwd, *cwd.parents):
        for candidate in (anchor, anchor / REL_STUDY_DIR):
            if all((candidate / sample).is_dir() for sample in SAMPLES):
                return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate Examples/YeGhasemmi2018. Start Jupyter in the ORCA "
        "repository or in the YeGhasemmi2018 study directory."
    )


STUDY_DIR = locate_study_directory()
RUN_DIR = STUDY_DIR / "final_simulation_runs"
SAMPLE_HPC_ROOTS = {
    sample: sorted((STUDY_DIR / sample).glob("results_csv_hpc*"))
    for sample in SAMPLES
}
RESULT_ROOTS = [
    *(root for sample in SAMPLES for root in SAMPLE_HPC_ROOTS[sample]),
    RUN_DIR / "results_csv_hpc",
    RUN_DIR / "results_csv",
]
POINT_DIR_NAMES = ("point_values", "")
LINE_SUBDIR = "vector_line_postprocessors"
EXODUS_DIR = RUN_DIR / "results_exodus"

print(f"Study directory: {STUDY_DIR}")
print(f"Optional legacy run directory: {RUN_DIR}")
print("CSV search order:")
for root in RESULT_ROOTS:
    print(f"  {root}")


## Consolidated analysis artifact checks

This check keeps the consolidated narrative and the machine-readable Table 2 ranking tied to the notebook. It validates both files before the simulation campaign audit runs. Cyclic (97-series) and shut-in (98-series) cases remain outside the Table 2 ranking because their loading histories differ from the eleven-stage experiment.


In [ ]:
PROJECT_ROOT = STUDY_DIR.parents[1]
INDEPENDENT_ANALYSIS_DIR = PROJECT_ROOT / "doc" / "independent_analysis"
ANALYSIS_ARTIFACTS = {
    "consolidated_analysis": INDEPENDENT_ANALYSIS_DIR / "CONSOLIDATED_ANALYSIS_2026-08-18.md",
    "table2_ranking": INDEPENDENT_ANALYSIS_DIR / "TABLE2_ERROR_ACCURACY_RANKING.csv",
}

missing_analysis_artifacts = {
    name: path for name, path in ANALYSIS_ARTIFACTS.items() if not path.is_file()
}
if missing_analysis_artifacts:
    missing = "\n".join(
        f"  {name}: {path}" for name, path in missing_analysis_artifacts.items()
    )
    raise FileNotFoundError(f"Missing consolidated analysis artifacts:\n{missing}")

consolidated_text = ANALYSIS_ARTIFACTS["consolidated_analysis"].read_text(encoding="utf-8")
if not consolidated_text.startswith("# ORCA 4.0 consolidated analysis"):
    raise ValueError(
        f"Unexpected consolidated-analysis heading in {ANALYSIS_ARTIFACTS['consolidated_analysis']}"
    )

TABLE2_RANKING = pd.read_csv(ANALYSIS_ARTIFACTS["table2_ranking"])
required_ranking_columns = {
    "sample", "rank_within_sample", "case", "run_status",
    "comparable_for_ranking", "stages_reached", "total_stages",
    "mean_nrmse_pct", "accuracy_pct_100_minus_mean_nrmse",
    "source_csv", "notes",
}
missing_ranking_columns = sorted(required_ranking_columns.difference(TABLE2_RANKING.columns))
if missing_ranking_columns:
    raise KeyError(
        f"{ANALYSIS_ARTIFACTS['table2_ranking'].name}: missing columns {missing_ranking_columns}"
    )

ranking_samples = set(TABLE2_RANKING["sample"].dropna().astype(str))
if ranking_samples != set(SAMPLES):
    raise ValueError(
        f"Ranking sample coverage is {sorted(ranking_samples)}, expected {sorted(SAMPLES)}"
    )
if TABLE2_RANKING.duplicated(["sample", "case"]).any():
    duplicates = TABLE2_RANKING.loc[
        TABLE2_RANKING.duplicated(["sample", "case"], keep=False), ["sample", "case"]
    ]
    raise ValueError(f"Duplicate sample/case rows in ranking:\n{duplicates.to_string(index=False)}")

comparable = (
    TABLE2_RANKING["comparable_for_ranking"]
    .astype(str).str.strip().str.lower().eq("true")
)
complete = TABLE2_RANKING["run_status"].astype(str).str.lower().eq("complete")
if not comparable.equals(complete):
    raise ValueError("Ranking comparability must be true exactly for complete runs")
if TABLE2_RANKING.loc[comparable, [
    "rank_within_sample", "mean_nrmse_pct", "accuracy_pct_100_minus_mean_nrmse"
]].isna().any().any():
    raise ValueError("Every comparable run must have rank, error, and accuracy values")
if TABLE2_RANKING.loc[~comparable, "rank_within_sample"].notna().any():
    raise ValueError("Partial runs must not receive a within-sample rank")

reported_accuracy = pd.to_numeric(
    TABLE2_RANKING.loc[comparable, "accuracy_pct_100_minus_mean_nrmse"], errors="raise"
)
expected_accuracy = 100.0 - pd.to_numeric(
    TABLE2_RANKING.loc[comparable, "mean_nrmse_pct"], errors="raise"
)
if not np.allclose(reported_accuracy, expected_accuracy, rtol=0.0, atol=1.0e-6):
    raise ValueError("Ranking accuracy does not equal 100 - mean nRMSE")

artifact_status = pd.DataFrame([
    {
        "artifact": name,
        "path": str(path.relative_to(PROJECT_ROOT)),
        "exists": path.is_file(),
        "size_bytes": path.stat().st_size,
        "rows": len(TABLE2_RANKING) if name == "table2_ranking" else np.nan,
    }
    for name, path in ANALYSIS_ARTIFACTS.items()
])
display(artifact_status)
display(
    TABLE2_RANKING.groupby(["sample", "run_status"], observed=False)
    .size().unstack(fill_value=0).reindex(SAMPLES, fill_value=0)
)


## Campaign discovery and data audit

The following cell reads the manifest, all scalar result tables, all centerline filenames, and all available Exodus filenames. It does not infer success from file existence: scalar coverage is measured against each input file's requested final time.


In [ ]:
manifest_path = RUN_DIR / "run_manifest.csv"
if manifest_path.is_file():
    MANIFEST = pd.read_csv(
        manifest_path,
        dtype={"run_id": str, "family": str, "model": str, "case_id": str},
    )
    MANIFEST["mesh_size_mm"] = pd.to_numeric(
        MANIFEST["mesh_size_mm"], errors="raise"
    ).astype(int)
    MANIFEST["source"] = "publication_manifest"
else:
    MANIFEST = pd.DataFrame(columns=[
        "run_id", "family", "model", "case_id", "mesh_size_mm", "source"
    ])
    print(f"Legacy manifest not found at {manifest_path}; using per-sample HPC results only.")

# Register every per-sample HPC CSV, including runs not present in run_manifest.csv.
HPC_POINT_PATHS = {}
hpc_manifest_rows = []
known_run_ids = set(MANIFEST["run_id"])
for sample in SAMPLES:
    for root in SAMPLE_HPC_ROOTS[sample]:
        for csv_path in sorted(root.glob("*.csv")):
            run_id = csv_path.stem
            previous = HPC_POINT_PATHS.get(run_id)
            if previous is not None and previous != csv_path:
                warnings.warn(f"Duplicate HPC result stem {run_id}: {previous} and {csv_path}")
                continue
            HPC_POINT_PATHS[run_id] = csv_path
            if run_id not in known_run_ids:
                hpc_manifest_rows.append({
                    "run_id": run_id,
                    "family": sample,
                    "model": "HPC/Rorqual",
                    "case_id": run_id,
                    "mesh_size_mm": np.nan,
                    "source": "sample_hpc",
                })
                known_run_ids.add(run_id)
if hpc_manifest_rows:
    MANIFEST = pd.concat(
        [MANIFEST, pd.DataFrame(hpc_manifest_rows)], ignore_index=True
    )


def configured_end_time(input_path):
    if input_path is None or not input_path.is_file():
        return np.nan
    text = input_path.read_text(errors="replace")
    values = re.findall(
        r"(?m)^\s*end_time\s*=\s*([0-9.eE+-]+)\s*(?:#.*)?$",
        text,
    )
    if not values:
        return np.nan
    return float(values[-1])


def input_path_for(record):
    if record.source == "sample_hpc":
        stem = record.run_id.removesuffix("_hpc")
        candidates = [
            STUDY_DIR / record.family / f"{stem}.i",
            STUDY_DIR / record.family / f"{record.run_id}.i",
        ]
        return next((path for path in candidates if path.is_file()), None)
    path = RUN_DIR / f"{record.run_id}.i"
    return path if path.is_file() else None


def relative_result_path(path):
    if path is None:
        return None
    for base in (STUDY_DIR, RUN_DIR):
        try:
            return str(path.relative_to(base))
        except ValueError:
            pass
    return str(path)


def point_candidates(run_id):
    paths = []
    direct_hpc_path = HPC_POINT_PATHS.get(run_id)
    if direct_hpc_path is not None:
        paths.append(direct_hpc_path)
    for priority, root in enumerate(RESULT_ROOTS):
        for subdir in POINT_DIR_NAMES:
            path = root / subdir / f"{run_id}.csv" if subdir else root / f"{run_id}.csv"
            if path.is_file() and path not in paths:
                paths.append(path)
    return paths


def read_best_point_result(run_id):
    readable = []
    errors = []
    for path in point_candidates(run_id):
        try:
            frame = pd.read_csv(path, low_memory=False)
            if "time" not in frame.columns:
                raise ValueError("missing required 'time' column")
            time = pd.to_numeric(frame["time"], errors="coerce")
            finite_time = time[np.isfinite(time)]
            last_time = float(finite_time.max()) if len(finite_time) else -np.inf
            root_priority = next(
                (i for i, root in enumerate(RESULT_ROOTS) if root in path.parents),
                len(RESULT_ROOTS),
            )
            readable.append(
                ((last_time, len(frame), -root_priority), path, frame)
            )
        except Exception as exc:
            errors.append(f"{path}: {exc}")
    if not readable:
        return None, None, errors
    _, path, frame = max(readable, key=lambda item: item[0])
    return path, frame, errors


POINT_DATA = {}
POINT_PATHS = {}
READ_ERRORS = {}
for run_id in MANIFEST["run_id"]:
    path, frame, errors = read_best_point_result(run_id)
    if frame is not None:
        POINT_DATA[run_id] = frame
        POINT_PATHS[run_id] = path
    if errors:
        READ_ERRORS[run_id] = errors

# Prefer filenames from the HPC copy, while allowing the local output tree as fallback.
profile_by_name = {}
for root in RESULT_ROOTS:
    directory = root / LINE_SUBDIR
    if directory.is_dir():
        for path in directory.glob("*.csv"):
            profile_by_name.setdefault(path.name, path)

profile_pattern = re.compile(
    r"^(?P<run_id>.+)_centerline_(?P<axis>[xyz])_(?P<step>\d+)\.csv$"
)
profile_rows = []
for path in profile_by_name.values():
    match = profile_pattern.match(path.name)
    if match:
        profile_rows.append(
            {
                "run_id": match.group("run_id"),
                "axis": match.group("axis"),
                "step": int(match.group("step")),
                "path": path,
            }
        )
LINE_FILES = pd.DataFrame(
    profile_rows, columns=["run_id", "axis", "step", "path"]
)
if not LINE_FILES.empty:
    LINE_FILES = LINE_FILES.sort_values(
        ["run_id", "axis", "step"]
    ).reset_index(drop=True)

EXODUS_FILES = {}
if EXODUS_DIR.is_dir():
    for path in EXODUS_DIR.glob("*.e"):
        EXODUS_FILES[path.stem] = path

status_rows = []
for record in MANIFEST.itertuples(index=False):
    run_id = record.run_id
    input_path = input_path_for(record)
    expected_end = configured_end_time(input_path)
    data = POINT_DATA.get(run_id)

    if data is None:
        row_count = 0
        column_count = 0
        first_time = np.nan
        last_time = np.nan
        completion = 0.0
        result_status = "missing"
    else:
        time = pd.to_numeric(data["time"], errors="coerce")
        finite_time = time[np.isfinite(time)]
        row_count = len(data)
        column_count = max(0, len(data.columns) - 1)
        first_time = float(finite_time.min()) if len(finite_time) else np.nan
        last_time = float(finite_time.max()) if len(finite_time) else np.nan
        completion = (
            100.0 * last_time / expected_end
            if np.isfinite(last_time) and np.isfinite(expected_end) and expected_end > 0
            else np.nan
        )
        result_status = (
            "complete"
            if np.isfinite(last_time)
            and np.isfinite(expected_end)
            and last_time >= expected_end - COMPLETE_TOLERANCE
            else "partial"
        )

    line_count = (
        int((LINE_FILES["run_id"] == run_id).sum())
        if not LINE_FILES.empty
        else 0
    )
    point_path = POINT_PATHS.get(run_id)
    status_rows.append(
        {
            "run_id": run_id,
            "family": record.family,
            "model": record.model,
            "case_id": record.case_id,
            "mesh_mm": record.mesh_size_mm,
            "status": result_status,
            "rows": row_count,
            "scalar_columns": column_count,
            "first_time_s": first_time,
            "last_time_s": last_time,
            "expected_end_s": expected_end,
            "coverage_pct": completion,
            "line_profile_files": line_count,
            "exodus": run_id in EXODUS_FILES,
            "point_csv": relative_result_path(point_path),
        }
    )

STATUS = pd.DataFrame(status_rows)

print(f"Registered runs: {len(MANIFEST)}")
print(f"Per-sample HPC histories discovered: {len(HPC_POINT_PATHS)}")
print(f"Readable scalar histories: {len(POINT_DATA)}")
print(f"Unique centerline CSVs: {len(LINE_FILES):,}")
print(f"Exodus files: {len(EXODUS_FILES)}")
if READ_ERRORS:
    warnings.warn(
        f"{sum(map(len, READ_ERRORS.values()))} candidate CSV(s) could not be read; "
        "inspect READ_ERRORS for details."
    )


In [ ]:
audit_columns = [
    "run_id", "family", "model", "case_id", "mesh_mm", "status",
    "rows", "scalar_columns", "last_time_s", "expected_end_s",
    "coverage_pct", "line_profile_files", "exodus", "point_csv",
]
display(
    STATUS[audit_columns].style.format(
        {
            "last_time_s": "{:,.6g}",
            "expected_end_s": "{:,.6g}",
            "coverage_pct": "{:.1f}%",
        },
        na_rep="—",
    ).map(
        lambda value: (
            "background-color: #d8f3dc"
            if value == "complete"
            else "background-color: #ffe5b4"
            if value == "partial"
            else "background-color: #ffd6d6"
            if value == "missing"
            else ""
        ),
        subset=["status"],
    )
)

fig, ax = plt.subplots(figsize=(13, max(8, 0.22 * len(STATUS))))
plot_status = STATUS.sort_values(["family", "model", "case_id", "mesh_mm"])
colors = plot_status["status"].map(
    {"complete": "#2a9d8f", "partial": "#e9c46a", "missing": "#e76f51"}
)
ax.barh(
    plot_status["run_id"],
    plot_status["coverage_pct"].fillna(0).clip(upper=100),
    color=colors,
)
ax.axvline(100, color="black", linewidth=1, linestyle=":")
ax.set_xlim(0, 105)
ax.set_xlabel("Configured simulation time reached (%)")
ax.set_ylabel("")
ax.set_title(f"All {len(STATUS)} registered result histories — observed coverage")
ax.invert_yaxis()
fig.tight_layout()
plt.show()

display(
    STATUS.groupby(["family", "status"], observed=False)
    .size()
    .unstack(fill_value=0)
    .reindex(SAMPLES, fill_value=0)
)


## Plotting and inspection helpers

The core panels use readable units. The complete scalar panels then plot every remaining scalar field in the CSV schema, including constitutive diagnostics, reactions, aperture/permeability measures, flow balance, and bulk/fault stresses. Solid lines are 3 mm meshes; dashed lines are 5 mm meshes.


In [ ]:
CORE_SPECS = {
    "injection_pressure_pp": ("Injection pressure", 1.0e-6, "MPa", False),
    "differential_stress_reaction_mpa_pp": (
        "Reaction differential stress", 1.0, "MPa", False
    ),
    "effective_normal_compression_mpa_pp": (
        "Effective normal compression", 1.0, "MPa", False
    ),
    "shear_traction_magnitude_pa": ("Shear traction magnitude", 1.0e-6, "MPa", False),
    "czm_shear_slip_mm_pp": ("CZM shear slip", 1.0, "mm", False),
    "czm_normal_dilation_paper_mm_pp": ("CZM normal dilation", 1.0, "mm", False),
    "hydraulic_aperture_um_pp": ("Hydraulic aperture", 1.0, "µm", False),
    "fracture_permeability_pp": ("Fracture permeability", 1.0, "m²", True),
    "flow_rate_validation_ml_min_pp": (
        "Validation flow rate", 1.0, "mL/min", False
    ),
    "fracture_pressure_mean_pp": ("Mean fracture pressure", 1.0e-6, "MPa", False),
    "friction_coefficient_effective_pp": (
        "Effective friction coefficient", 1.0, "—", False
    ),
    "flow_mass_imbalance_fraction_pp": (
        "Flow mass-imbalance fraction", 1.0, "—", False
    ),
}
CORE_COLUMNS = list(CORE_SPECS)

case_keys = (
    MANIFEST[["family", "model", "case_id"]]
    .drop_duplicates()
    .apply(lambda row: " / ".join(row.astype(str)), axis=1)
    .tolist()
)
palette = plt.colormaps["tab20"](np.linspace(0, 1, max(1, len(case_keys))))
CASE_COLORS = dict(zip(case_keys, palette))


def selected_run_ids(sample):
    selection = MANUAL_RESULT_FILES.get(sample)
    available = MANIFEST.loc[MANIFEST["family"].eq(sample), "run_id"].tolist()
    if selection is None:
        return available
    requested = list(dict.fromkeys(Path(str(name)).stem for name in selection))
    missing = [run_id for run_id in requested if run_id not in available]
    if missing:
        warnings.warn(
            f"{sample}: requested result file(s) were not discovered: {missing}"
        )
    return [run_id for run_id in requested if run_id in available]


def sample_manifest(sample):
    selected = selected_run_ids(sample)
    order = {run_id: index for index, run_id in enumerate(selected)}
    frame = MANIFEST.loc[MANIFEST["run_id"].isin(selected)].copy()
    frame["_selection_order"] = frame["run_id"].map(order)
    return frame.sort_values("_selection_order").drop(columns="_selection_order")


def run_label(run_id):
    row = MANIFEST.loc[MANIFEST["run_id"].eq(run_id)].iloc[0]
    if row["source"] == "sample_hpc":
        return run_id
    return f"{row['model']} {row['case_id']} — {row['mesh_size_mm']} mm"


def run_style(run_id):
    row = MANIFEST.loc[MANIFEST["run_id"].eq(run_id)].iloc[0]
    key = f"{row['family']} / {row['model']} / {row['case_id']}"
    is_hpc = row["source"] == "sample_hpc"
    mesh_mm = pd.to_numeric(row["mesh_size_mm"], errors="coerce")
    return {
        "color": CASE_COLORS[key],
        "linestyle": "-" if is_hpc or mesh_mm == 3 else "--",
        "linewidth": 1.45,
    }


def friendly_name(column):
    name = column
    for suffix in ("_pp", "_pa"):
        if name.endswith(suffix):
            name = name[: -len(suffix)]
    return name.replace("_", " ").strip().title()


def scalar_columns(sample):
    ordered = []
    for run_id in sample_manifest(sample)["run_id"]:
        data = POINT_DATA.get(run_id)
        if data is None:
            continue
        for column in data.columns:
            if column == "time" or column in ordered:
                continue
            if pd.api.types.is_numeric_dtype(data[column]):
                ordered.append(column)
    return ordered


def plot_columns(sample, columns, heading, per_figure=SCALAR_PLOTS_PER_FIGURE):
    run_ids = sample_manifest(sample)["run_id"].tolist()
    columns = [
        column for column in columns
        if any(column in POINT_DATA.get(run_id, pd.DataFrame()).columns for run_id in run_ids)
    ]
    if not columns:
        print(f"{sample}: no requested scalar columns are available.")
        return []

    figures = []
    ncols = 3
    for page_start in range(0, len(columns), per_figure):
        page = columns[page_start : page_start + per_figure]
        nrows = math.ceil(len(page) / ncols)
        fig, axes = plt.subplots(
            nrows, ncols, figsize=(16, 3.7 * nrows), squeeze=False
        )
        for ax, column in zip(axes.flat, page):
            spec = CORE_SPECS.get(
                column, (friendly_name(column), 1.0, "native CSV units", False)
            )
            title, scale, unit, log_y = spec
            plotted = 0
            for run_id in run_ids:
                data = POINT_DATA.get(run_id)
                if data is None or column not in data:
                    continue
                time = pd.to_numeric(data["time"], errors="coerce").to_numpy(float)
                values = (
                    pd.to_numeric(data[column], errors="coerce").to_numpy(float)
                    * scale
                )
                finite = np.isfinite(time) & np.isfinite(values)
                if log_y:
                    finite &= values > 0
                if not finite.any():
                    continue
                style = run_style(run_id)
                ax.plot(
                    time[finite], values[finite],
                    label=run_label(run_id), **style
                )
                plotted += 1
            ax.set_title(title, fontsize=10)
            ax.set_xlabel("Time (s)")
            ax.set_ylabel(unit)
            if log_y:
                ax.set_yscale("log")
            if plotted == 0:
                ax.text(
                    0.5, 0.5, "No finite values",
                    transform=ax.transAxes, ha="center", va="center"
                )
        for ax in axes.flat[len(page):]:
            ax.set_visible(False)

        handles, labels = [], []
        for run_id in run_ids:
            if run_id not in POINT_DATA:
                continue
            style = run_style(run_id)
            handles.append(
                Line2D([0], [0], label=run_label(run_id), **style)
            )
            labels.append(run_label(run_id))
        fig.legend(
            handles=handles,
            labels=labels,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.995),
            ncol=min(4, max(1, len(handles))),
            fontsize=8,
        )
        page_number = page_start // per_figure + 1
        page_count = math.ceil(len(columns) / per_figure)
        fig.suptitle(
            f"{sample} — {heading} ({page_number}/{page_count})",
            y=1.04,
            fontsize=14,
        )
        fig.tight_layout(rect=(0, 0, 1, 0.94))
        plt.show()
        figures.append(fig)
    return figures


def plot_core_results(sample):
    return plot_columns(sample, CORE_COLUMNS, "core response histories")


def plot_all_remaining_scalars(sample):
    remaining = [
        column for column in scalar_columns(sample)
        if column not in CORE_COLUMNS
    ]
    print(
        f"{sample}: {len(CORE_COLUMNS)} configured core fields and "
        f"{len(remaining)} additional scalar fields."
    )
    if not PLOT_ALL_SCALAR_COLUMNS:
        print(
            "Complete scalar plotting is disabled. Set "
            "PLOT_ALL_SCALAR_COLUMNS = True in the setup cell and rerun."
        )
        return []
    return plot_columns(
        sample,
        remaining,
        "all additional scalar histories",
    )


In [ ]:
def line_inventory(sample):
    run_ids = set(sample_manifest(sample)["run_id"])
    if LINE_FILES.empty:
        return pd.DataFrame(
            columns=["run_id", "axis", "files", "first_step", "last_step"]
        )
    selected = LINE_FILES.loc[LINE_FILES["run_id"].isin(run_ids)]
    if selected.empty:
        return pd.DataFrame(
            columns=["run_id", "axis", "files", "first_step", "last_step"]
        )
    return (
        selected.groupby(["run_id", "axis"], as_index=False)
        .agg(
            files=("path", "size"),
            first_step=("step", "min"),
            last_step=("step", "max"),
        )
        .sort_values(["run_id", "axis"])
    )


def profile_files(run_id, axis=None):
    if LINE_FILES.empty:
        return LINE_FILES.copy()
    selected = LINE_FILES.loc[LINE_FILES["run_id"].eq(run_id)]
    if axis is not None:
        selected = selected.loc[selected["axis"].eq(axis)]
    return selected.sort_values(["axis", "step"]).reset_index(drop=True)


def list_profile_files(run_id, axis=None):
    """Return the complete file catalog for a run (and optionally one axis)."""
    result = profile_files(run_id, axis).copy()
    if not result.empty:
        result["path"] = result["path"].map(
            lambda path: str(path.relative_to(RUN_DIR))
        )
    return result


PROFILE_SPECS = {
    "pore_pressure": ("Pore pressure", 1.0e-6, "MPa"),
    "disp_x": ("Displacement x", 1.0e6, "µm"),
    "disp_y": ("Displacement y", 1.0e6, "µm"),
    "disp_z": ("Displacement z", 1.0e6, "µm"),
}


def _profile_time(run_id, step):
    data = POINT_DATA.get(run_id)
    if data is None or step < 0 or step >= len(data):
        return np.nan
    return float(pd.to_numeric(data["time"], errors="coerce").iloc[step])


def _select_profile(run_id, axis, snapshot):
    files = profile_files(run_id, axis)
    if files.empty:
        return None
    if snapshot == "last":
        return files.iloc[-1]
    if snapshot == "first":
        return files.iloc[0]
    target = int(snapshot)
    return files.iloc[(files["step"] - target).abs().argmin()]


def plot_line_snapshot(
    run_id,
    snapshot="last",
    fields=("pore_pressure", "disp_x", "disp_y", "disp_z"),
):
    """Plot one available x/y/z centerline snapshot for a run."""
    if run_id not in set(MANIFEST["run_id"]):
        raise KeyError(f"Unknown run_id: {run_id}")
    available = profile_files(run_id)
    if available.empty:
        print(f"{run_id}: no centerline CSV files are present.")
        return None

    fig, axes = plt.subplots(
        len(fields), 3, figsize=(15, 3.1 * len(fields)), squeeze=False
    )
    for column_index, axis_name in enumerate(("x", "y", "z")):
        record = _select_profile(run_id, axis_name, snapshot)
        for row_index, field in enumerate(fields):
            ax = axes[row_index, column_index]
            if record is None:
                ax.text(
                    0.5, 0.5, f"No centerline-{axis_name} file",
                    transform=ax.transAxes, ha="center", va="center"
                )
                ax.set_axis_off()
                continue
            data = pd.read_csv(record["path"])
            if axis_name not in data or field not in data:
                ax.text(
                    0.5, 0.5, f"Missing {field}",
                    transform=ax.transAxes, ha="center", va="center"
                )
                continue
            coordinate = pd.to_numeric(data[axis_name], errors="coerce").to_numpy(float)
            values = pd.to_numeric(data[field], errors="coerce").to_numpy(float)
            title, scale, unit = PROFILE_SPECS.get(
                field, (friendly_name(field), 1.0, "native CSV units")
            )
            finite = np.isfinite(coordinate) & np.isfinite(values)
            order = np.argsort(coordinate[finite])
            ax.plot(
                coordinate[finite][order],
                values[finite][order] * scale,
                color="tab:blue",
                linewidth=1.5,
            )
            step = int(record["step"])
            time = _profile_time(run_id, step)
            time_text = f", t≈{time:.6g} s" if np.isfinite(time) else ""
            ax.set_title(
                f"{title} — centerline {axis_name}, file {step}{time_text}",
                fontsize=9,
            )
            ax.set_xlabel(f"{axis_name} coordinate (m)")
            ax.set_ylabel(unit)
    fig.suptitle(f"{run_id} — centerline snapshot", y=1.01, fontsize=13)
    fig.tight_layout()
    plt.show()
    return fig


def plot_latest_profiles(sample):
    if LINE_FILES.empty:
        print(f"{sample}: no centerline CSV files are present.")
        return []
    run_ids = sample_manifest(sample)["run_id"].tolist()
    available_runs = [
        run_id for run_id in run_ids
        if not profile_files(run_id).empty
    ]
    if not available_runs:
        print(f"{sample}: no centerline CSV files are present.")
        return []
    figures = []
    for run_id in available_runs:
        figures.append(plot_line_snapshot(run_id, snapshot="last"))
    return figures


def plot_line_evolution(
    run_id,
    axis="x",
    field="pore_pressure",
    snapshots=10,
):
    """Plot evenly spaced files from the complete centerline catalog."""
    files = profile_files(run_id, axis)
    if files.empty:
        print(f"{run_id}: no centerline-{axis} CSV files are present.")
        return None
    count = min(max(1, int(snapshots)), len(files))
    positions = np.unique(
        np.linspace(0, len(files) - 1, count).round().astype(int)
    )
    selected = files.iloc[positions]

    title, scale, unit = PROFILE_SPECS.get(
        field, (friendly_name(field), 1.0, "native CSV units")
    )
    fig, ax = plt.subplots(figsize=(11, 6))
    colors = plt.colormaps["viridis"](np.linspace(0, 1, len(selected)))
    for color, (_, record) in zip(colors, selected.iterrows()):
        data = pd.read_csv(record["path"])
        if axis not in data or field not in data:
            continue
        coordinate = pd.to_numeric(data[axis], errors="coerce").to_numpy(float)
        values = pd.to_numeric(data[field], errors="coerce").to_numpy(float)
        finite = np.isfinite(coordinate) & np.isfinite(values)
        order = np.argsort(coordinate[finite])
        step = int(record["step"])
        time = _profile_time(run_id, step)
        label = f"file {step}"
        if np.isfinite(time):
            label += f", t≈{time:.6g} s"
        ax.plot(
            coordinate[finite][order],
            values[finite][order] * scale,
            color=color,
            linewidth=1.25,
            label=label,
        )
    ax.set_title(f"{run_id} — {title} along centerline {axis}")
    ax.set_xlabel(f"{axis} coordinate (m)")
    ax.set_ylabel(unit)
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    plt.show()
    return fig


def show_sample_inventory(sample):
    selected = selected_run_ids(sample)
    rows = STATUS.loc[STATUS["run_id"].isin(selected), audit_columns]
    available_count = int(STATUS["family"].eq(sample).sum())
    print(f"{sample}: plotting {len(selected)} of {available_count} registered result(s)")
    display(
        rows.style.format(
            {
                "last_time_s": "{:,.6g}",
                "expected_end_s": "{:,.6g}",
                "coverage_pct": "{:.1f}%",
            },
            na_rep="—",
        )
    )
    profiles = line_inventory(sample)
    print(f"{sample} centerline inventory")
    display(profiles if not profiles.empty else pd.DataFrame({"message": ["No centerline CSV files present"]}))
    exodus_rows = [
        {
            "run_id": run_id,
            "path": str(EXODUS_FILES[run_id].relative_to(RUN_DIR)),
            "size_GiB": EXODUS_FILES[run_id].stat().st_size / 1024**3,
        }
        for run_id in sample_manifest(sample)["run_id"]
        if run_id in EXODUS_FILES
    ]
    print(f"{sample} Exodus inventory")
    display(
        pd.DataFrame(exodus_rows)
        if exodus_rows
        else pd.DataFrame({"message": ["No Exodus files present"]})
    )


def point_result(run_id):
    """Return the complete, unmodified scalar DataFrame for one run."""
    return POINT_DATA[run_id]


def available_results(sample=None):
    """List every registered scalar result, including files not selected for plots."""
    rows = STATUS if sample is None else STATUS.loc[STATUS["family"].eq(sample)]
    return rows[audit_columns].reset_index(drop=True)


## SWT1 — dedicated results block

This block plots every discovered SWT1 result because `MANUAL_RESULT_FILES["SWT1"]` is `None`.


In [ ]:
show_sample_inventory("SWT1")


In [ ]:
plot_core_results("SWT1")


In [ ]:
plot_all_remaining_scalars("SWT1")


In [ ]:
plot_latest_profiles("SWT1")


## SWT2 — dedicated results block

This block plots every discovered SWT2 result because `MANUAL_RESULT_FILES["SWT2"]` is `None`.


In [ ]:
show_sample_inventory("SWT2")


In [ ]:
plot_core_results("SWT2")


In [ ]:
plot_all_remaining_scalars("SWT2")


In [ ]:
plot_latest_profiles("SWT2")


## SWS3 — dedicated results block

This block plots every discovered SWS3 result because `MANUAL_RESULT_FILES["SWS3"]` is `None`.


In [ ]:
show_sample_inventory("SWS3")


In [ ]:
plot_core_results("SWS3")


In [ ]:
plot_all_remaining_scalars("SWS3")


In [ ]:
plot_latest_profiles("SWS3")


## SWS4 — dedicated results block

This block plots every discovered SWS4 result because `MANUAL_RESULT_FILES["SWS4"]` is `None`. The separate validation cell overlays all packaged Ye et al. histories.


In [ ]:
show_sample_inventory("SWS4")


In [ ]:
plot_core_results("SWS4")


In [ ]:
plot_all_remaining_scalars("SWS4")


### SWS4 experimental validation overlay

The repository currently packages experimental histories for SWS4 only. Mechanical simulation curves start at 55 s, and the simulated slip/dilation histories are referenced to their 55 s values to match the published validation convention.


In [ ]:
VALIDATION_FILES = {
    "injection_pressure": "Ye2018_SW4_Injection_pressure_Vs_time.csv",
    "differential_stress": "Ye2018_SW4_Differential_Stress_Vs_time.csv",
    "effective_normal": "Ye2018_SW4_normal_stress_Vs_time.csv",
    "shear_stress": "Ye2018_SW4_shear_stress_Vs_time.csv",
    "shear_slip": "Ye2018_SW4_shear_slip_Vs_time.csv",
    "normal_dilation": "Ye2018_SW4_normal_dilation_Vs_time.csv",
    "permeability": "Ye2018_SW4_frac_perm_Vs_time.csv",
    "flow_rate": "Ye2018_SW4_flow_rate_Vs_time.csv",
}
VALIDATION_SPECS = {
    "injection_pressure": ("injection_pressure_pp", 1.0e-6, False, False, "Injection pressure (MPa)"),
    "differential_stress": ("differential_stress_reaction_mpa_pp", 1.0, False, True, "Differential stress (MPa)"),
    "effective_normal": ("effective_normal_compression_mpa_pp", 1.0, False, True, "Effective normal compression (MPa)"),
    "shear_stress": ("shear_traction_magnitude_pa", 1.0e-6, False, True, "Shear stress (MPa)"),
    "shear_slip": ("czm_shear_slip_mm_pp", 1.0, True, True, "Shear slip from 55 s (mm)"),
    "normal_dilation": ("czm_normal_dilation_paper_mm_pp", 1.0, True, True, "Normal dilation from 55 s (mm)"),
    "permeability": ("fracture_permeability_pp", 1.0, False, False, "Fracture permeability (m²)"),
    "flow_rate": ("flow_rate_validation_ml_min_pp", 1.0, False, False, "Flow rate (mL/min)"),
}


def load_validation_curves():
    directory = STUDY_DIR / "SWS4" / "SWS4"
    curves = {}
    for key, filename in VALIDATION_FILES.items():
        path = directory / filename
        if not path.is_file():
            continue
        frame = pd.read_csv(path, header=None, names=["time", "value"])
        frame = frame.apply(pd.to_numeric, errors="coerce").dropna()
        curves[key] = frame.sort_values("time")
    return curves


def validation_model_series(data, key):
    column, scale, reference_at_55, mechanical, _ = VALIDATION_SPECS[key]
    if column not in data:
        return np.array([]), np.array([])
    time = pd.to_numeric(data["time"], errors="coerce").to_numpy(float)
    raw = pd.to_numeric(data[column], errors="coerce").to_numpy(float) * scale
    finite = np.isfinite(time) & np.isfinite(raw)
    mask = finite & (time > 0)
    if mechanical:
        mask &= time >= 55.0
    values = raw[mask]
    if reference_at_55 and finite.any():
        ordered = np.argsort(time[finite])
        base = np.interp(55.0, time[finite][ordered], raw[finite][ordered])
        values = values - base
    return time[mask], values


def plot_sws4_validation():
    curves = load_validation_curves()
    if not curves:
        print("No SWS4 validation curves found.")
        return None
    run_ids = sample_manifest("SWS4")["run_id"].tolist()
    fig, axes = plt.subplots(4, 2, figsize=(16, 17), squeeze=False)
    for ax, (key, curve) in zip(axes.flat, curves.items()):
        _, _, _, _, ylabel = VALIDATION_SPECS[key]
        ax.scatter(
            curve["time"], curve["value"],
            s=14, color="black", label="Ye et al.", zorder=10
        )
        for run_id in run_ids:
            data = POINT_DATA.get(run_id)
            if data is None:
                continue
            time, values = validation_model_series(data, key)
            if not len(time):
                continue
            ax.plot(
                time, values,
                label=run_label(run_id),
                **run_style(run_id),
            )
        ax.set_title(key.replace("_", " ").title())
        ax.set_xlabel("Time (s)")
        ax.set_ylabel(ylabel)
        if key == "permeability":
            ax.set_yscale("log")
        ax.legend(fontsize=7, ncol=2)
    fig.suptitle("SWS4 — every run against all packaged validation histories", y=1.005, fontsize=14)
    fig.tight_layout()
    plt.show()
    return fig


plot_sws4_validation()


In [ ]:
plot_latest_profiles("SWS4")


## Direct access to every result file

All scalar tables, including every discovered per-sample HPC CSV, are loaded in **POINT_DATA** whether or not they are selected for plotting. Use **available_results(sample)** to list their names and **point_result(run_id)** to obtain a complete DataFrame without plotting or truncation. Use **list_profile_files(run_id, axis)** to list every centerline file, **plot_line_snapshot(...)** for any file number, and **plot_line_evolution(...)** for evenly spaced snapshots.

Examples are left commented so a full “Run All” does not duplicate figures.


In [ ]:
# Complete scalar table (returns every row and column):
# display(available_results("SWS3"))
# data = point_result("91_05_sw3_bbfast_paperjrc_resc1p65_kernel_SV_biot0p6_hpc")
# display(data)

# Complete centerline file catalog:
# display(list_profile_files("SWS3_BBFast_case84_00_mesh5", axis="x"))

# Any requested centerline file number (nearest available file is selected):
# plot_line_snapshot("SWS3_BBFast_case84_00_mesh5", snapshot=2500)

# Evolution using evenly spaced files from the complete catalog:
# plot_line_evolution(
#     "SWS3_BBFast_case84_00_mesh5",
#     axis="x",
#     field="pore_pressure",
#     snapshots=12,
# )


## Individual inspection of every HPC result file

This final block discovers every CSV under the applicable `results_csv_hpc*`
directories, including runs not enabled in the comparison cells above. It processes
one file at a time and gives each file its own paginated figures containing every
finite numeric result column. It also displays per-column statistics. Set
`HPC_DISPLAY_FULL_TABLE = True` to display every raw row and column for each file.


In [ ]:
# Controls for the individual HPC-file inspection below.
HPC_SCALARS_PER_FIGURE = 12
HPC_PLOT_EVERY_NUMERIC_COLUMN = True
HPC_DISPLAY_COLUMN_STATISTICS = True
HPC_DISPLAY_FULL_TABLE = False  # Can create very large notebook outputs when True.


def discover_individual_hpc_files():
    """Return every HPC point-value CSV in scope for this notebook."""
    roots = []
    if 'HPC_RESULTS' in globals():
        roots.append(Path(HPC_RESULTS))
    if 'SAMPLE_HPC_ROOTS' in globals():
        roots.extend(
            Path(root)
            for sample_roots in SAMPLE_HPC_ROOTS.values()
            for root in sample_roots
        )
    files = {
        path.resolve(): path.resolve()
        for root in roots if root.is_dir()
        for path in root.glob('*.csv')
        if path.is_file()
    }
    return sorted(files.values(), key=lambda path: (path.parent.parent.name, path.name))


def read_individual_hpc_file(path):
    """Read one file without retaining every HPC DataFrame in memory."""
    frame = pd.read_csv(path)
    if 'time' not in frame.columns:
        raise KeyError(f'{path.name}: missing required time column')
    for column in frame.columns:
        frame[column] = pd.to_numeric(frame[column], errors='coerce')
    frame = (
        frame.dropna(subset=['time'])
        .sort_values('time')
        .drop_duplicates('time', keep='last')
        .reset_index(drop=True)
    )
    return frame


def individual_hpc_numeric_columns(frame):
    columns = []
    for column in frame.columns:
        if column == 'time':
            continue
        values = pd.to_numeric(frame[column], errors='coerce')
        if np.isfinite(values.to_numpy(float)).any():
            columns.append(column)
    return columns


def individual_hpc_column_title(column):
    if 'friendly_name' in globals():
        return friendly_name(column)
    return column.replace('_', ' ').strip().title()


def plot_individual_hpc_file(path, frame, per_figure=HPC_SCALARS_PER_FIGURE):
    """Plot every numeric history from one HPC CSV, never overlaying other runs."""
    columns = individual_hpc_numeric_columns(frame)
    if not columns:
        print(f'{path.name}: no finite numeric result columns')
        return 0
    if not HPC_PLOT_EVERY_NUMERIC_COLUMN:
        print(f'{path.name}: scalar plotting disabled by HPC_PLOT_EVERY_NUMERIC_COLUMN')
        return 0

    page_count = math.ceil(len(columns) / per_figure)
    for page_index, page_start in enumerate(range(0, len(columns), per_figure), start=1):
        page = columns[page_start:page_start + per_figure]
        ncols = 3
        nrows = math.ceil(len(page) / ncols)
        fig, axes = plt.subplots(
            nrows, ncols, figsize=(16, 3.7 * nrows), squeeze=False
        )
        time = frame['time'].to_numpy(float)
        for ax, column in zip(axes.flat, page):
            values = frame[column].to_numpy(float)
            finite = np.isfinite(time) & np.isfinite(values)
            ax.plot(time[finite], values[finite], color='tab:blue', linewidth=1.25)
            ax.set_title(individual_hpc_column_title(column), fontsize=9)
            ax.set_xlabel('Time (s)')
            ax.set_ylabel(column, fontsize=8)
            # Permeability and transmissivity commonly span orders of magnitude.
            lower_name = column.lower()
            positive = values[finite]
            if (
                len(positive)
                and np.all(positive > 0)
                and ('permeability' in lower_name or 'transmissivity' in lower_name)
            ):
                ax.set_yscale('log')
            ax.tick_params(labelsize=8)
        for ax in axes.flat[len(page):]:
            ax.set_visible(False)
        fig.suptitle(
            f'{path.name} — individual HPC results ({page_index}/{page_count})',
            fontsize=13,
        )
        fig.tight_layout(rect=(0, 0, 1, 0.97))
        plt.show()
        plt.close(fig)
    return page_count


HPC_RESULT_FILES = discover_individual_hpc_files()
print(f'Found {len(HPC_RESULT_FILES)} HPC result file(s) for individual inspection.')

individual_hpc_summary_rows = []
for hpc_path in HPC_RESULT_FILES:
    print('\n' + '=' * 100)
    print(f'HPC result file: {hpc_path}')
    try:
        hpc_frame = read_individual_hpc_file(hpc_path)
    except Exception as exc:
        warnings.warn(f'Could not inspect {hpc_path}: {exc}')
        individual_hpc_summary_rows.append({
            'file': hpc_path.name,
            'sample': hpc_path.parent.parent.name,
            'status': f'ERROR: {exc}',
            'rows': 0,
            'numeric_result_columns': 0,
            'final_time_s': np.nan,
            'figure_pages': 0,
        })
        continue

    hpc_columns = individual_hpc_numeric_columns(hpc_frame)
    print(
        f'{len(hpc_frame):,} rows; {len(hpc_columns)} finite numeric result columns; '
        f't = {hpc_frame["time"].min():.6g}–{hpc_frame["time"].max():.6g} s'
    )

    if HPC_DISPLAY_COLUMN_STATISTICS:
        statistics = hpc_frame[hpc_columns].describe().T
        statistics.index.name = 'result_column'
        display(statistics)
    if HPC_DISPLAY_FULL_TABLE:
        with pd.option_context('display.max_rows', None, 'display.max_columns', None):
            display(hpc_frame)

    pages = plot_individual_hpc_file(hpc_path, hpc_frame)
    individual_hpc_summary_rows.append({
        'file': hpc_path.name,
        'sample': hpc_path.parent.parent.name,
        'status': 'OK',
        'rows': int(len(hpc_frame)),
        'numeric_result_columns': int(len(hpc_columns)),
        'final_time_s': float(hpc_frame['time'].max()),
        'figure_pages': int(pages),
    })
    del hpc_frame

INDIVIDUAL_HPC_SUMMARY = pd.DataFrame(individual_hpc_summary_rows)
display(INDIVIDUAL_HPC_SUMMARY)
